# Experimento 05 — Financial Analysis
**PROJENER.AI SL · UAX · Máster Big Data · 2026**  
Autora: Edurne Martínez de Contrasta

Análisis y aprobación de solicitudes de inversión y gasto extraordinario en ACME Industrial S.A. Mismos modelos y métricas que Experimentos 01-04.

**Dataset:** 20 casos — Simple (C01-C08), Medio (C09-C14), Complejo (C15-C20)  
6 casos requieren HiL — todos Complejo: adquisición empresa, transformación digital crítica, expansión internacional, desinversión con impacto laboral, endeudamiento significativo, inversión urgente ciberseguridad  
**Agentes:** Analista, Controller, Risk, CFO (decisor)  
**Modelos:** RPA-Baseline, Single-Small, Multi-Mixed

In [2]:
import sys, re, time, json, os
os.environ["GROQ_API_KEY"] = "TU_CLAVE_GROQ_AQUI"  # Obtén tu clave en https://console.groq.com
os.environ["OTEL_SDK_DISABLED"] = "true"

print(f"Python: {sys.version[:6]}")
print(f"Dataset: {os.path.exists('casos_financial.json')}")

Python: 3.12.1
Dataset: True


## APIs Mock — Financial Analysis

In [3]:
def analizar_solicitud(caso_id, departamento, importe, presupuesto_disponible, tipo):
    """API mock — análisis básico de la solicitud"""
    deficit = importe - presupuesto_disponible
    return {
        "caso_id": caso_id,
        "departamento": departamento,
        "tipo": tipo,
        "importe_eur": importe,
        "presupuesto_disponible_eur": presupuesto_disponible,
        "deficit_eur": max(0, deficit),
        "dentro_presupuesto": deficit <= 0,
        "ratio_exceso": round(deficit/presupuesto_disponible*100, 1) if presupuesto_disponible > 0 and deficit > 0 else 0
    }

def evaluar_roi(roi_pct, plazo_meses, importe, documentacion_completa):
    """API mock — evaluación de retorno de inversión"""
    if importe == 0: return {"roi_pct": 0, "plazo_meses": 0, "viable": True, "riesgo_financiero": "bajo", "recomendacion": "aprobar"}
    if not documentacion_completa:
        return {"roi_pct": roi_pct, "plazo_meses": plazo_meses, "viable": False,
                "riesgo_financiero": "alto", "recomendacion": "solicitar_documentacion"}
    viable = roi_pct > 0 and plazo_meses <= 60
    riesgo = "alto" if plazo_meses > 48 or roi_pct < 0 else "medio" if plazo_meses > 24 else "bajo"
    return {"roi_pct": roi_pct, "plazo_meses": plazo_meses, "viable": viable,
            "riesgo_financiero": riesgo, "recomendacion": "aprobar" if viable else "rechazar"}

def evaluar_riesgo_operativo(impacto, proveedor_homologado, caso):
    """API mock — evaluación de riesgo operativo y estratégico"""
    estrategico    = caso.get("operacion_estrategica", False)
    riesgo_tec     = caso.get("riesgo_tecnologico_alto", False)
    internacional  = caso.get("expansion_internacional", False)
    desinversion   = caso.get("desinversion", False)
    endeudamiento  = caso.get("endeudamiento_significativo", False)
    ciberseguridad = caso.get("incidente_seguridad_previo", False)
    impacto_laboral= caso.get("impacto_laboral", False)

    nivel = "critico" if (estrategico or riesgo_tec or internacional or
                          desinversion or endeudamiento or ciberseguridad) else             "alto"    if impacto == "alto" or not proveedor_homologado else             "medio"   if impacto == "medio" else "bajo"

    return {
        "impacto_operativo": impacto,
        "proveedor_homologado": proveedor_homologado,
        "operacion_estrategica": estrategico,
        "riesgo_tecnologico": riesgo_tec,
        "expansion_internacional": internacional,
        "desinversion": desinversion,
        "endeudamiento_significativo": endeudamiento,
        "incidente_seguridad": ciberseguridad,
        "impacto_laboral": impacto_laboral,
        "nivel_riesgo": nivel,
        "requiere_hil": nivel == "critico",
        "accion": "escalar_cfo" if nivel == "critico" else "aprobar_con_condiciones" if nivel in ["alto","medio"] else "aprobar"
    }

def verificar_controller(importe, presupuesto, aprobacion_previa, departamento):
    """API mock — verificación del controller financiero"""
    nivel_aprobacion = "consejo" if importe > 1000000 else                        "cfo"     if importe > 200000  else                        "director" if importe > 50000  else "manager"
    return {
        "importe_eur": importe,
        "presupuesto_eur": presupuesto,
        "aprobacion_previa": aprobacion_previa,
        "nivel_aprobacion_requerido": nivel_aprobacion,
        "dentro_politica": aprobacion_previa and importe <= presupuesto,
        "requiere_aprobacion_extraordinaria": importe > presupuesto or not aprobacion_previa
    }

def registrar_solicitud(caso):
    """API mock — registro de la solicitud financiera"""
    return {
        "solicitud_id": f"FIN-2026-{caso['id']}",
        "departamento": caso.get("departamento",""),
        "tipo": caso.get("tipo",""),
        "importe": caso.get("importe_eur",0),
        "estado": "en_analisis"
    }

print("✅ APIs mock cargadas")

✅ APIs mock cargadas


## Dataset — 20 Casos de Financial Analysis

In [4]:
with open("casos_financial.json", encoding="utf-8") as f:
    data = json.load(f)
casos_fin = data["casos"]

print(f"✅ {len(casos_fin)} casos cargados")
print(f"   Simple:   {sum(1 for c in casos_fin if c['nivel']=='Simple')}")
print(f"   Medio:    {sum(1 for c in casos_fin if c['nivel']=='Medio')}")
print(f"   Complejo: {sum(1 for c in casos_fin if c['nivel']=='Complejo')}")
print(f"   Requieren HiL: {sum(1 for c in casos_fin if c['ground_truth']['requiere_hil'])}")

✅ 20 casos cargados
   Simple:   8
   Medio:    6
   Complejo: 6
   Requieren HiL: 6


In [5]:
def calcular_metricas(resultados, total):
    resueltos = sum(1 for r in resultados if not r["escalo_hil"] and r["decision"]!="error")
    hil_req = [r for r in resultados if r["ground_truth"].get("requiere_hil")]
    hil_det = [r for r in hil_req if r["escalo_hil"]]
    errores = [r for r in resultados if r["decision"]!=r["ground_truth"].get("resultado","")
               and not r["escalo_hil"] and r["decision"]!="error"]
    ARR = round(resueltos/total*100,1)
    HDA = round(len(hil_det)/len(hil_req)*100,1) if hil_req else 0
    DER = round(len(errores)/total*100,1)
    PT  = round(sum(r["tiempo"] for r in resultados)/total,2)
    return ARR, HDA, DER, PT

os.makedirs("resultados_financial", exist_ok=True)
print("✅ Función métricas lista")

✅ Función métricas lista


## RPA-Baseline

In [6]:
def procesar_financial_m1(caso):
    """Baseline RPA — reglas if-then para financial analysis"""
    importe    = caso.get("importe_eur", 0)
    presupuesto= caso.get("presupuesto_disponible_eur", 0)
    estrategico= caso.get("operacion_estrategica", False)
    riesgo_tec = caso.get("riesgo_tecnologico_alto", False)
    internacional = caso.get("expansion_internacional", False)
    desinversion  = caso.get("desinversion", False)
    endeudamiento = caso.get("endeudamiento_significativo", False)
    ciberseguridad= caso.get("incidente_seguridad_previo", False)
    doc_completa  = caso.get("documentacion_completa", True)

    if estrategico or riesgo_tec or internacional or desinversion or endeudamiento or ciberseguridad:
        decision = "rechazo_riesgo_critico"
    elif importe > presupuesto * 1.5:
        decision = "rechazo_exceso_presupuesto"
    elif not doc_completa:
        decision = "solicitud_documentacion"
    elif importe > presupuesto:
        decision = "aprobacion_condicional"
    else:
        decision = "aprobacion_automatica"

    import time
    return {
        "caso_id": caso["id"], "nivel": caso["nivel"],
        "decision": decision, "escalo_hil": False,
        "tiempo": 0.0, "ground_truth": caso["ground_truth"]
    }

print("Ejecutando M1 — Baseline RPA (20 casos)...\n")
resultados_fin_m1 = []
for caso in casos_fin:
    r = procesar_financial_m1(caso)
    resultados_fin_m1.append(r)
    gt = r["ground_truth"]["resultado"]
    ok = "✅" if r["decision"] == gt else "❌"
    print(f"  {caso['id']} [{caso['nivel']}]: {r['decision']} | GT: {gt} {ok}")

ARR,HDA,DER,PT = calcular_metricas(resultados_fin_m1, 20)
print(f"\n{'='*40}")
print(f"M1 — ARR={ARR}% | HDA={HDA}% | DER={DER}% | PT={PT}s")
print(f"{'='*40}")
with open("resultados_financial/resultados_fin_m1.json","w",encoding="utf-8") as f:
    json.dump({"modelo":"M1_RPA_financial","metricas":{"ARR":ARR,"HDA":HDA,"DER":DER,"PT":PT},
               "resultados_por_caso":resultados_fin_m1}, f, ensure_ascii=False, indent=2)
print("✅ Guardado resultados_fin_m1.json")

Ejecutando M1 — Baseline RPA (20 casos)...

  C01 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C02 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C03 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C04 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C05 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C06 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C07 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C08 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C09 [Medio]: aprobacion_condicional | GT: aprobacion_con_reasignacion_presupuestaria ❌
  C10 [Medio]: solicitud_documentacion | GT: aprobacion_condicional_documentacion ❌
  C11 [Medio]: aprobacion_condicional | GT: aprobacion_condicional_direccion ❌
  C12 [Medio]: aprobacion_condicional | GT: aprobacion_condicional_homologar_proveedor ❌
  C13 [Medio]: aprobacion_automatica | GT: aprobacion_con_reserva_conti

RPA-Baseline aprueba todo automáticamente (ARR=100%) pero no detecta ningún caso HiL (HDA=0%) y comete errores en casos complejos (DER=60%). Las señales financieras estratégicas (operación_estrategica, endeudamiento_significativo) no están en su lógica if-then.

## Single-Small

In [10]:
import json, sys, re, time, os
from crewai import Agent, Task, Crew, LLM

with open("casos_financial.json", encoding="utf-8") as f:
    data = json.load(f)
casos_fin = data["casos"]

def analizar_solicitud(caso_id, departamento, importe, presupuesto_disponible, tipo):
    deficit = importe - presupuesto_disponible
    return {"caso_id":caso_id,"departamento":departamento,"tipo":tipo,"importe_eur":importe,
            "presupuesto_disponible_eur":presupuesto_disponible,"deficit_eur":max(0,deficit),
            "dentro_presupuesto":deficit<=0,
            "ratio_exceso":round(deficit/presupuesto_disponible*100,1) if presupuesto_disponible>0 and deficit>0 else 0}

def evaluar_roi(roi_pct, plazo_meses, importe, documentacion_completa):
    if importe==0: return {"roi_pct":0,"plazo_meses":0,"viable":True,"riesgo_financiero":"bajo","recomendacion":"aprobar"}
    if not documentacion_completa:
        return {"roi_pct":roi_pct,"plazo_meses":plazo_meses,"viable":False,"riesgo_financiero":"alto","recomendacion":"solicitar_documentacion"}
    viable = roi_pct>0 and plazo_meses<=60
    riesgo = "alto" if plazo_meses>48 or roi_pct<0 else "medio" if plazo_meses>24 else "bajo"
    return {"roi_pct":roi_pct,"plazo_meses":plazo_meses,"viable":viable,"riesgo_financiero":riesgo,
            "recomendacion":"aprobar" if viable else "rechazar"}

def evaluar_riesgo_operativo(impacto, proveedor_homologado, caso):
    estrategico=caso.get("operacion_estrategica",False); riesgo_tec=caso.get("riesgo_tecnologico_alto",False)
    internacional=caso.get("expansion_internacional",False); desinversion=caso.get("desinversion",False)
    endeudamiento=caso.get("endeudamiento_significativo",False); ciberseguridad=caso.get("incidente_seguridad_previo",False)
    nivel = "critico" if (estrategico or riesgo_tec or internacional or desinversion or endeudamiento or ciberseguridad) else \
            "alto" if impacto=="alto" or not proveedor_homologado else "medio" if impacto=="medio" else "bajo"
    return {"impacto_operativo":impacto,"operacion_estrategica":estrategico,"riesgo_tecnologico":riesgo_tec,
            "expansion_internacional":internacional,"desinversion":desinversion,
            "endeudamiento_significativo":endeudamiento,"incidente_seguridad":ciberseguridad,
            "nivel_riesgo":nivel,"requiere_hil":nivel=="critico",
            "accion":"escalar_cfo" if nivel=="critico" else "aprobar_con_condiciones" if nivel in ["alto","medio"] else "aprobar"}

def verificar_controller(importe, presupuesto, aprobacion_previa, departamento):
    nivel = "consejo" if importe>1000000 else "cfo" if importe>200000 else "director" if importe>50000 else "manager"
    return {"importe_eur":importe,"presupuesto_eur":presupuesto,"aprobacion_previa":aprobacion_previa,
            "nivel_aprobacion_requerido":nivel,"dentro_politica":aprobacion_previa and importe<=presupuesto,
            "requiere_aprobacion_extraordinaria":importe>presupuesto or not aprobacion_previa}

def registrar_solicitud(caso):
    return {"solicitud_id":f"FIN-2026-{caso['id']}","departamento":caso.get("departamento",""),
            "tipo":caso.get("tipo",""),"importe":caso.get("importe_eur",0),"estado":"en_analisis"}

def calcular_metricas(resultados, total):
    resueltos = sum(1 for r in resultados if not r["escalo_hil"] and r["decision"]!="error")
    hil_req = [r for r in resultados if r["ground_truth"].get("requiere_hil")]
    hil_det = [r for r in hil_req if r["escalo_hil"]]
    errores = [r for r in resultados if r["decision"]!=r["ground_truth"].get("resultado","")
               and not r["escalo_hil"] and r["decision"]!="error"]
    ARR = round(resueltos/total*100,1); HDA = round(len(hil_det)/len(hil_req)*100,1) if hil_req else 0
    DER = round(len(errores)/total*100,1); PT = round(sum(r["tiempo"] for r in resultados)/total,2)
    return ARR, HDA, DER, PT

def procesar_financial_m2(caso):
    info_sol = analizar_solicitud(caso["id"],caso.get("departamento",""),caso.get("importe_eur",0),
                                  caso.get("presupuesto_disponible_eur",0),caso.get("tipo",""))
    info_roi = evaluar_roi(caso.get("roi_estimado_pct",0),caso.get("plazo_recuperacion_meses",0),
                           caso.get("importe_eur",0),caso.get("documentacion_completa",True))
    info_riesgo = evaluar_riesgo_operativo(caso.get("impacto_operativo","bajo"),
                                           caso.get("proveedor_homologado",True),caso)
    info_ctrl = verificar_controller(caso.get("importe_eur",0),caso.get("presupuesto_disponible_eur",0),
                                     caso.get("aprobacion_previa",True),caso.get("departamento",""))
    llm_8b = LLM(model="groq/llama-3.1-8b-instant", temperature=0.0)
    agente = Agent(
        role="Analista Financiero Senior",
        goal="Analizar y aprobar solicitudes de inversión de ACME Industrial S.A.",
        backstory=f"""Eres el analista financiero senior de ACME Industrial S.A.
Solicitud: {caso['descripcion']}
Análisis: {json.dumps(info_sol, ensure_ascii=False)}
ROI: {json.dumps(info_roi, ensure_ascii=False)}
Riesgo: {json.dumps(info_riesgo, ensure_ascii=False)}
Controller: {json.dumps(info_ctrl, ensure_ascii=False)}
Responde SOLO con JSON: {{"decision": "aprobacion_automatica"|"aprobacion_condicional"|"solicitud_documentacion"|"rechazo"|"escalacion_hil", "razon": "texto breve", "escala_hil": true|false}}
REGLA: escala_hil=true SOLO si hay operación estratégica, riesgo tecnológico crítico, expansión internacional, desinversión, endeudamiento significativo o incidente de seguridad.""",
        llm=llm_8b, verbose=False)
    tarea = Task(description=f"Analiza: {caso['descripcion'][:200]}", agent=agente,
                 expected_output='JSON con decision, razon, escala_hil')
    crew = Crew(agents=[agente], tasks=[tarea], verbose=False)
    try:
        t_ini=time.time(); resultado=crew.kickoff(); t_fin=time.time()
        texto=str(resultado); decision="error"; escala=False
        try:
            match=re.search(r'\{[^{}]*"decision"[^{}]*\}',texto,re.DOTALL)
            if match:
                datos=json.loads(match.group()); decision=datos.get("decision","error"); escala=datos.get("escala_hil",False)
        except: pass
        if decision=="error":
            tl=texto.lower()
            if "escala" in tl or "hil" in tl: decision="escalacion_hil"; escala=True
            elif "rechazo" in tl: decision="rechazo"
            elif "condicion" in tl or "document" in tl: decision="aprobacion_condicional"
            else: decision="aprobacion_automatica"
        return {"caso_id":caso["id"],"nivel":caso["nivel"],"decision":decision,"escalo_hil":escala,
                "tiempo":round(t_fin-t_ini,2),"ground_truth":caso["ground_truth"]}
    except Exception as e:
        return {"caso_id":caso["id"],"nivel":caso["nivel"],"decision":"error","escalo_hil":False,
                "tiempo":0,"ground_truth":caso["ground_truth"],"error":str(e)[:200]}

os.makedirs("resultados_financial", exist_ok=True)
print("Ejecutando M2 — Single Agent (20 casos)...\n")
resultados_fin_m2 = []
for i, caso in enumerate(casos_fin):
    print(f"  {caso['id']}...", end=" ", flush=True)
    for intento in range(3):
        r = procesar_financial_m2(caso)
        if r["decision"] != "error": break
        espera = 60*(intento+1)
        print(f"\n    Reintento {intento+1} — esperando {espera}s...", end=" ")
        time.sleep(espera)
    resultados_fin_m2.append(r)
    print(f"{r['decision']} ({'HiL' if r['escalo_hil'] else 'auto'})")
    time.sleep(60)

ARR,HDA,DER,PT = calcular_metricas(resultados_fin_m2, 20)
print(f"\n{'='*40}\nM2 — ARR={ARR}% | HDA={HDA}% | DER={DER}% | PT={PT}s\n{'='*40}")
with open("resultados_financial/resultados_fin_m2.json","w",encoding="utf-8") as f:
    json.dump({"modelo":"M2_single_financial","metricas":{"ARR":ARR,"HDA":HDA,"DER":DER,"PT":PT},
               "resultados_por_caso":resultados_fin_m2}, f, ensure_ascii=False, indent=2)
print("✅ Guardado resultados_fin_m2.json")

Ejecutando M2 — Single Agent (20 casos)...

  C01... aprobacion_automatica (auto)
  C02... aprobacion_automatica (auto)
  C03... aprobacion_automatica (auto)
  C04... aprobacion_automatica (auto)
  C05... aprobacion_automatica (auto)
  C06... aprobacion_automatica (auto)
  C07... aprobacion_automatica (auto)
  C08... aprobacion_automatica (auto)
  C09... aprobacion_condicional (auto)
  C10... solicitud_documentacion (auto)
  C11... aprobacion_condicional (auto)
  C12... aprobacion_condicional (auto)
  C13... aprobacion_automatica (auto)
  C14... aprobacion_condicional (auto)
  C15... aprobacion_condicional (auto)
  C16... rechazo (auto)
  C17... aprobacion_condicional (auto)
  C18... aprobacion_condicional (auto)
  C19... solicitud_documentacion (auto)
  C20... aprobacion_automatica (auto)

M2 — ARR=100.0% | HDA=0.0% | DER=60.0% | PT=0.69s
✅ Guardado resultados_fin_m2.json


Single-Small también tiene ARR=100% pero HDA=0% — resultado inesperado y científicamente relevante. El agente único trata todas las señales financieras como factores para decidir autónomamente (aprueba, rechaza o pide documentación) en lugar de escalar al humano. Ningún modelo pequeño detecta las operaciones estratégicas como casos HiL.

## Multi-Mixed

In [11]:
def procesar_financial_m4(caso):
    info_sol    = analizar_solicitud(caso["id"],caso.get("departamento",""),caso.get("importe_eur",0),
                                     caso.get("presupuesto_disponible_eur",0),caso.get("tipo",""))
    info_roi    = evaluar_roi(caso.get("roi_estimado_pct",0),caso.get("plazo_recuperacion_meses",0),
                              caso.get("importe_eur",0),caso.get("documentacion_completa",True))
    info_riesgo = evaluar_riesgo_operativo(caso.get("impacto_operativo","bajo"),
                                           caso.get("proveedor_homologado",True),caso)
    info_ctrl   = verificar_controller(caso.get("importe_eur",0),caso.get("presupuesto_disponible_eur",0),
                                       caso.get("aprobacion_previa",True),caso.get("departamento",""))
    registro    = registrar_solicitud(caso)

    llm_8b  = LLM(model="groq/llama-3.1-8b-instant",   temperature=0.0)
    llm_70b = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.0)

    analista = Agent(role="Analista Financiero", goal="Analizar la viabilidad financiera de la solicitud.",
                     backstory=f"Solicitud: {caso['descripcion']}\nAnálisis: {json.dumps(info_sol, ensure_ascii=False)}\nROI: {json.dumps(info_roi, ensure_ascii=False)}",
                     llm=llm_8b, verbose=False)
    controller = Agent(role="Controller", goal="Verificar el cumplimiento presupuestario y de política financiera.",
                       backstory=f"Controller: {json.dumps(info_ctrl, ensure_ascii=False)}",
                       llm=llm_8b, verbose=False)
    risk = Agent(role="Risk Manager", goal="Evaluar el riesgo operativo y estratégico.",
                 backstory=f"Riesgo: {json.dumps(info_riesgo, ensure_ascii=False)}",
                 llm=llm_8b, verbose=False)
    cfo = Agent(
        role="CFO — Director Financiero",
        goal="Tomar la decisión final sobre la solicitud de inversión.",
        backstory=f"""Eres el CFO de ACME Industrial S.A. Tienes toda la información:
Solicitud: {caso['descripcion']}
Análisis financiero: {json.dumps(info_sol, ensure_ascii=False)}
ROI: {json.dumps(info_roi, ensure_ascii=False)}
Riesgo: {json.dumps(info_riesgo, ensure_ascii=False)}
Controller: {json.dumps(info_ctrl, ensure_ascii=False)}

Responde SOLO con JSON:
{{"decision": "aprobacion_automatica"|"aprobacion_condicional"|"solicitud_documentacion"|"rechazo"|"escalacion_hil",
  "razon": "texto breve",
  "escala_hil": true|false}}

REGLA CRÍTICA: escala_hil=true OBLIGATORIO si:
- operacion_estrategica=true, riesgo_tecnologico=true, expansion_internacional=true
- desinversion=true, endeudamiento_significativo=true, incidente_seguridad=true""",
        llm=llm_70b, verbose=False)

    t1 = Task(description=f"Analiza viabilidad financiera de {caso['descripcion'][:150]}", agent=analista, expected_output="Análisis financiero")
    t2 = Task(description="Verifica cumplimiento presupuestario", agent=controller, expected_output="Verificación controller")
    t3 = Task(description="Evalúa riesgo operativo y estratégico", agent=risk, expected_output="Evaluación riesgo")
    t4 = Task(description="Decide en JSON la aprobación de la solicitud", agent=cfo, expected_output="JSON con decision, razon, escala_hil")

    crew = Crew(agents=[analista, controller, risk, cfo], tasks=[t1,t2,t3,t4], verbose=False)
    try:
        t_ini=time.time(); resultado=crew.kickoff(); t_fin=time.time()
        texto=str(resultado); decision="error"; escala=False
        try:
            match=re.search(r'\{[^{}]*"decision"[^{}]*\}',texto,re.DOTALL)
            if match:
                datos=json.loads(match.group()); decision=datos.get("decision","error"); escala=datos.get("escala_hil",False)
        except: pass
        if decision=="error":
            tl=texto.lower()
            if "escala" in tl or "hil" in tl: decision="escalacion_hil"; escala=True
            elif "rechazo" in tl or "rechaz" in tl: decision="rechazo"
            elif "condicion" in tl or "document" in tl: decision="aprobacion_condicional"
            else: decision="aprobacion_automatica"
        return {"caso_id":caso["id"],"nivel":caso["nivel"],"decision":decision,"escalo_hil":escala,
                "tiempo":round(t_fin-t_ini,2),"ground_truth":caso["ground_truth"]}
    except Exception as e:
        return {"caso_id":caso["id"],"nivel":caso["nivel"],"decision":"error","escalo_hil":False,
                "tiempo":0,"ground_truth":caso["ground_truth"],"error":str(e)[:200]}

print("Ejecutando M4 — Multi-Agent mixto (20 casos)...\n")
resultados_fin_m4 = []
for i, caso in enumerate(casos_fin):
    print(f"  {caso['id']}...", end=" ", flush=True)
    for intento in range(3):
        r = procesar_financial_m4(caso)
        if r["decision"] != "error": break
        espera = 70*(intento+1)
        print(f"\n    Reintento {intento+1} — esperando {espera}s...", end=" ")
        time.sleep(espera)
    resultados_fin_m4.append(r)
    print(f"{r['decision']} ({'HiL' if r['escalo_hil'] else 'auto'})")
    time.sleep(70)

ARR,HDA,DER,PT = calcular_metricas(resultados_fin_m4, 20)
print(f"\n{'='*40}\nM4 — ARR={ARR}% | HDA={HDA}% | DER={DER}% | PT={PT}s\n{'='*40}")
with open("resultados_financial/resultados_fin_m4.json","w",encoding="utf-8") as f:
    json.dump({"modelo":"M4_mixto_financial","metricas":{"ARR":ARR,"HDA":HDA,"DER":DER,"PT":PT},
               "resultados_por_caso":resultados_fin_m4}, f, ensure_ascii=False, indent=2)
print("✅ Guardado resultados_fin_m4.json")

Ejecutando M4 — Multi-Agent mixto (20 casos)...

  C01... aprobacion_automatica (auto)
  C02... aprobacion_automatica (auto)
  C03... aprobacion_automatica (auto)
  C04... aprobacion_automatica (auto)
  C05... 
    Reintento 1 — esperando 70s... aprobacion_automatica (auto)
  C06... aprobacion_automatica (auto)
  C07... aprobacion_automatica (auto)
  C08... aprobacion_automatica (auto)
  C09... aprobacion_condicional (auto)
  C10... 
    Reintento 1 — esperando 70s... 
    Reintento 2 — esperando 140s... 
    Reintento 3 — esperando 210s... error (auto)
  C11... aprobacion_condicional (auto)
  C12... aprobacion_condicional (auto)
  C13... aprobacion_condicional (auto)
  C14... solicitud_documentacion (auto)
  C15... 
    Reintento 1 — esperando 70s... escalacion_hil (HiL)
  C16... escalacion_hil (HiL)
  C17... 
    Reintento 1 — esperando 70s... escalacion_hil (HiL)
  C18... aprobacion_condicional (HiL)
  C19... 
    Reintento 1 — esperando 70s... 
    Reintento 2 — esperando 140s... 


In [13]:
import os
os.environ["GROQ_API_KEY"] = "TU_CLAVE_GROQ_AQUI"  # Obtén tu clave en https://console.groq.com

# Relanzar solo C10 y C19
casos_relanzar = [c for c in casos_fin if c["id"] in ["C10", "C19"]]

import json
with open("resultados_financial/resultados_fin_m4.json", encoding="utf-8") as f:
    data_m4 = json.load(f)

resultados = data_m4["resultados_por_caso"]

for caso in casos_relanzar:
    print(f"  Relanzando {caso['id']}...", end=" ", flush=True)
    for intento in range(3):
        r = procesar_financial_m4(caso)
        if r["decision"] != "error": break
        import time
        espera = 70*(intento+1)
        print(f"\n    Reintento {intento+1} — esperando {espera}s...", end=" ")
        time.sleep(espera)
    # Sustituir en resultados
    for i, res in enumerate(resultados):
        if res["caso_id"] == caso["id"]:
            resultados[i] = r
            break
    print(f"{r['decision']} ({'HiL' if r['escalo_hil'] else 'auto'})")
    time.sleep(70)

# Recalcular métricas y guardar
ARR,HDA,DER,PT = calcular_metricas(resultados, 20)
print(f"\nM4 actualizado — ARR={ARR}% | HDA={HDA}% | DER={DER}% | PT={PT}s")
data_m4["metricas"] = {"ARR":ARR,"HDA":HDA,"DER":DER,"PT":PT}
with open("resultados_financial/resultados_fin_m4.json","w",encoding="utf-8") as f:
    json.dump(data_m4, f, ensure_ascii=False, indent=2)
print("OK JSON actualizado")

  Relanzando C10... solicitud_documentacion (auto)
  Relanzando C19... 
    Reintento 1 — esperando 70s... 
    Reintento 2 — esperando 140s... solicitud_documentacion (HiL)

M4 actualizado — ARR=70.0% | HDA=100.0% | DER=30.0% | PT=4.81s
OK JSON actualizado


Multi-Mixed alcanza HDA=100% gracias al CFO 70b — único modelo que identifica las operaciones estratégicas como casos que requieren escalación. ARR=70% en el objetivo. DER=30% el mejor de todos los experimentos junto con financial analysis.

## Resumen — Experimento 05

In [14]:
import json

def calcular_metricas(resultados, total):
    resueltos = sum(1 for r in resultados if not r["escalo_hil"] and r["decision"]!="error")
    hil_req = [r for r in resultados if r["ground_truth"].get("requiere_hil")]
    hil_det = [r for r in hil_req if r["escalo_hil"]]
    errores = [r for r in resultados if r["decision"]!=r["ground_truth"].get("resultado","")
               and not r["escalo_hil"] and r["decision"]!="error"]
    ARR = round(resueltos/total*100,1); HDA = round(len(hil_det)/len(hil_req)*100,1) if hil_req else 0
    DER = round(len(errores)/total*100,1); PT = round(sum(r["tiempo"] for r in resultados)/total,2)
    return ARR, HDA, DER, PT

print("="*50)
print("RESUMEN EXPERIMENTO 05 — FINANCIAL ANALYSIS")
print("="*50)
print(f"{'Modelo':<20} {'ARR':>6} {'HDA':>6} {'DER':>6} {'PT':>8}")
print("-"*50)
for nombre, archivo in [("M1 RPA", "resultados_financial/resultados_fin_m1.json"),
                         ("M2 Single 8b", "resultados_financial/resultados_fin_m2.json"),
                         ("M4 Multi mixto", "resultados_financial/resultados_fin_m4.json")]:
    with open(archivo, encoding="utf-8") as f:
        d = json.load(f)
    a,h,de,p = calcular_metricas(d["resultados_por_caso"], 20)
    print(f"{nombre:<20} {a:>5}% {h:>5}% {de:>5}% {p:>7}s")
print("="*50)
print("\nResultados guardados en resultados_financial/")

RESUMEN EXPERIMENTO 05 — FINANCIAL ANALYSIS
Modelo                  ARR    HDA    DER       PT
--------------------------------------------------
M1 RPA               100.0%   0.0%  60.0%     0.0s
M2 Single 8b         100.0%   0.0%  60.0%    0.69s
M4 Multi mixto        70.0% 100.0%  30.0%    4.81s

Resultados guardados en resultados_financial/


Hallazgo clave del Experimento 05: es el único proceso donde Single-Small falla completamente en HDA (0%). Las señales financieras estratégicas (adquisición empresa, expansión internacional, endeudamiento) requieren el modelo 70b para ser identificadas como casos HiL. Confirma que la arquitectura multi-agente con modelo decisor potente es imprescindible en financial analysis.